# FIN-001 — 01 · Data Audit

Data Gate evidence: what the raw data is, its quality issues, and the leakage-safe split. Companion to `docs/data_audit.md`. Run from the `notebooks/` folder.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # repo root importable
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
pd.set_option('display.max_columns', 20)

## 1. Load raw data and profile it

In [2]:
from src.preprocessing import load_raw
raw = load_raw()
print('shape:', raw.shape)
print('nulls:', raw.isna().sum().to_dict())
print('exact duplicate rows:', int(raw.duplicated().sum()))
print('unique descriptions:', raw['description'].nunique())
raw.head(5)

shape: (68000, 2)
nulls: {'description': 0, 'category': 0}
exact duplicate rows: 22298
unique descriptions: 45697


,description,category
0,[debit] NORTHWESTERN MUTUALEFT PYMT PPD ID: 99...,Insurance
1,[debit] MTG PMT PENFED CU,Mortgage
2,[debit] BETMGM 147 PARK BLVD UNION CITY 94587 ...,Entertainment
3,[debit] REPUBLIC SERVICES 222 MISSION CT CHICA...,Utilities
4,[debit] PYPL*365 MARKET,Groceries


In [3]:
# Class balance
raw['category'].value_counts().sort_index()

category
Education         4000
Entertainment     4000
Fees              4000
Groceries         4000
Healthcare        4000
Income            4000
Insurance         4000
Mortgage          4000
Personal Care     4000
Rent              4000
Restaurants       4000
Shopping          4000
Subscription      4000
Transfer          4000
Transportation    4000
Travel            4000
Utilities         4000
Name: count, dtype: int64

In [4]:
# Debit/credit prefix and short-text nature
pref = raw['description'].str.extract(r'^\[(debit|credit)\]')[0]
print('prefix counts:', pref.value_counts(dropna=False).to_dict())
ntok = raw['description'].str.replace(r'^\[(debit|credit)\]','',regex=True).str.split().apply(len)
print('desc token length: median %d, p95 %d, max %d' % (ntok.median(), ntok.quantile(.95), ntok.max()))

prefix counts: {'debit': 60007, 'credit': 7993}
desc token length: median 3, p95 8, max 13


**EDA conclusions.** 68k rows, 2 columns, no nulls, 17 balanced categories. Two things stand out: (1) ~33% exact-duplicate rows and (2) descriptions are very short and carry random per-transaction identifiers (store numbers, ZIPs, PPD IDs). Both are handled in cleaning so the reported score measures generalization, not memorization.

## 2. Normalization — keep merchant words, drop random IDs

In [5]:
from src.features import normalize_description
for d in ['[debit] PUBLIX 7860 UNIVERSITY LN OAKLAND 94601-4574 CA USA',
          '[credit] US GOVERNMENT DIR DEP PPD ID: 547185326']:
    print(repr(d))
    print('  ->', repr(normalize_description(d)))

'[debit] PUBLIX 7860 UNIVERSITY LN OAKLAND 94601-4574 CA USA'
  -> 'flag_debit publix <num> university ln oakland <num> <num> ca usa'
'[credit] US GOVERNMENT DIR DEP PPD ID: 547185326'
  -> 'flag_credit us government dir dep ppd id <num>'


## 3. Clean + stratified split, and verify no leakage

In [6]:
from src.preprocessing import clean, make_split
df = make_split(clean(raw))
print('rows after clean:', len(df))
print(df['split'].value_counts().to_dict())
# LEAKAGE CHECK: no description may appear in more than one split
cross = df.groupby('description')['split'].nunique()
print('descriptions crossing split boundary:', int((cross>1).sum()))
assert (cross==1).all(), 'LEAKAGE: a description is in >1 split'

rows after clean: 45692
{'train': 31984, 'valid': 6854, 'test': 6854}
descriptions crossing split boundary: 0


In [7]:
# Stratification preserved?
import pandas as pd
overall = df['category'].value_counts(normalize=True)
test = df[df.split=='test']['category'].value_counts(normalize=True)
pd.DataFrame({'overall':overall,'test':test}).round(3).sort_index()

,overall,test
category,,
Education,0.046,0.046
Entertainment,0.055,0.055
Fees,0.030,0.029
Groceries,0.078,0.078
Healthcare,0.056,0.056
Income,0.049,0.049
Insurance,0.068,0.068
Mortgage,0.037,0.037
Personal Care,0.057,0.057


**Data Gate status: GREEN.** Source + license documented (`data/README.md`), quality issues logged with decisions (`docs/data_audit.md`), split matches the real prediction setting and is boundary-verified above, preprocessing is fit on train only (`reports/preprocessing_manifest.json`).